# Assignment -- Cedar Grove Public Library: Checkouts

**5 problems, basic -> medium.** Same core skills as class (pandas: loading, cleaning,
grouping; `requests`: calling a real public API and parsing JSON) applied to a brand-new
scenario and dataset.

**Estimated time:** 45-60 minutes

## The scenario

Cedar Grove Public Library tracks every book checkout in `data/checkouts.csv`: who
checked out which book, when it was due back, when (if ever) it was returned, and any
late fee charged. The head librarian has five questions.

## Setup

```bash
pip install -r requirements.txt
python scripts/generate_checkouts_data.py   # only if data/checkouts.csv isn't already there
jupyter lab Assignment_Starter.ipynb
```

Problem 4 calls a real public API ([Open Library](https://openlibrary.org/)) and needs an
open internet connection; its function includes a fallback so the assignment stays
completable even if that call fails.

Each problem names the exact variable you need to produce, and most have a small
"check yourself" cell with `assert` statements right after.

---
## Problem 1 (basic) -- Load and get oriented

Load `data/checkouts.csv` into `checkouts_df`, parsing `checkout_date`, `due_date`, and
`return_date` as real dates. Then answer: how many checkouts are there in total, and how
many have never been returned (i.e. `return_date` is missing)? Store these two numbers as
`n_total_checkouts` and `n_still_checked_out`.

In [1]:
import numpy as np
import pandas as pd
import requests

pd.set_option("display.max_columns", 20)

In [2]:
# TODO: load data/checkouts.csv, parsing the three date columns
checkouts_df = pd.read_csv("data/checkouts.csv", parse_dates = ["checkout_date","due_date","return_date"])
checkouts_df.dtypes
# TODO: how many rows total, and how many are missing return_date?
n_total_checkouts = len(checkouts_df)
n_still_checked_out = checkouts_df["return_date"].isnull().sum()

print(f"{n_total_checkouts} total checkouts, {n_still_checked_out} still checked out")

160 total checkouts, 48 still checked out


---
## Problem 2 (basic-medium) -- Clean the data, the right way for each column

A missing `return_date` here does **not** mean bad data -- it means the book is still
checked out, which is a completely normal, valid state. So this time, don't drop those
rows or invent a fake return date for them; instead, capture "returned or not" explicitly.

Build `checkouts_clean` from `checkouts_df` with:

- a new boolean column `is_returned`, `True` where `return_date` is present and `False`
  where it's missing
- `late_fee` missing filled with `0` (missing here means no fee was ever charged -- either
  the book came back on time, or a librarian waived the fee)

In [3]:
checkouts_clean = checkouts_df.copy()

# TODO: add an is_returned boolean column (True where return_date is present)
checkouts_clean["is_returned"] = checkouts_clean["return_date"].isnull()

# TODO: fill missing late_fee with 0
checkouts_clean["late_fee"] = checkouts_clean["late_fee"].fillna(0)

checkouts_clean

,checkout_id,member_id,book_title,genre,checkout_date,due_date,return_date,late_fee,is_returned
0,CHK-3001,MEM-0014,The Great Gatsby,Classic Fiction,2026-06-12,2026-07-03,2026-07-03,0.0,False
1,CHK-3002,MEM-0034,Jane Eyre,Gothic,2026-07-19,2026-08-09,NaT,0.0,True
2,CHK-3003,MEM-0060,The Catcher in the Rye,Classic Fiction,2026-01-02,2026-01-23,NaT,0.0,True
3,CHK-3004,MEM-0051,The Hobbit,Adventure,2026-06-16,2026-07-07,NaT,0.0,True
4,CHK-3005,MEM-0028,The Catcher in the Rye,Classic Fiction,2026-04-01,2026-04-22,2026-04-22,0.0,False
...,...,...,...,...,...,...,...,...,...
155,CHK-3156,MEM-0038,Brave New World,Dystopian,2026-02-13,2026-03-06,NaT,0.0,True
156,CHK-3157,MEM-0047,1984,Dystopian,2026-06-23,2026-07-14,2026-07-14,0.0,False
157,CHK-3158,MEM-0027,Moby Dick,Adventure,2026-02-14,2026-03-07,2026-03-16,0.0,False
158,CHK-3159,MEM-0055,1984,Dystopian,2026-06-29,2026-07-20,2026-07-20,0.0,False


In [4]:
checkouts_clean.dtypes

checkout_id              object
member_id                object
book_title               object
genre                    object
checkout_date    datetime64[ns]
due_date         datetime64[ns]
return_date      datetime64[ns]
late_fee                float64
is_returned                bool
dtype: object

---
## Problem 3 (medium) -- Which genre racks up the most late fees?

Using only **returned** books (`is_returned == True`), compute the average `late_fee` per
`genre`, sorted from highest to lowest, as `avg_late_fee_by_genre`.

In [5]:
# TODO: filter to returned books only, then average late_fee by genre, sorted descending

returned_only = checkouts_clean[checkouts_clean["is_returned"] == True]
avg_late_fee_by_genre = checkouts_clean.groupby("genre")["late_fee"].sum()/len(checkouts_clean[checkouts_clean["is_returned"] == True])

avg_late_fee_by_genre.sort_values(ascending=False)
# sorted(avg_late_fee_by_genre, reverse=True)
avg_late_fee_by_genre

genre
Adventure             0.223958
Classic Fiction       0.218750
Dystopian             0.307292
Gothic                0.135417
Historical Fiction    0.260417
Name: late_fee, dtype: float64

In [6]:
avg_late_fee_by_genre = checkouts_clean.groupby("genre")["late_fee"].mean()
avg_late_fee_by_genre


genre
Adventure             0.383929
Classic Fiction       0.218750
Dystopian             0.491667
Gothic                0.250000
Historical Fiction    0.446429
Name: late_fee, dtype: float64

In [7]:
checkouts_clean[checkouts_clean["is_returned"] == True]


,checkout_id,member_id,book_title,genre,checkout_date,due_date,return_date,late_fee,is_returned
1,CHK-3002,MEM-0034,Jane Eyre,Gothic,2026-07-19,2026-08-09,NaT,0.0,True
2,CHK-3003,MEM-0060,The Catcher in the Rye,Classic Fiction,2026-01-02,2026-01-23,NaT,0.0,True
3,CHK-3004,MEM-0051,The Hobbit,Adventure,2026-06-16,2026-07-07,NaT,0.0,True
6,CHK-3007,MEM-0048,War and Peace,Historical Fiction,2026-05-24,2026-06-14,NaT,0.0,True
10,CHK-3011,MEM-0031,War and Peace,Historical Fiction,2026-01-28,2026-02-18,NaT,0.0,True
11,CHK-3012,MEM-0060,Moby Dick,Adventure,2026-05-31,2026-06-21,NaT,0.0,True
14,CHK-3015,MEM-0005,The Great Gatsby,Classic Fiction,2026-07-12,2026-08-02,NaT,0.0,True
19,CHK-3020,MEM-0003,Frankenstein,Gothic,2026-05-26,2026-06-16,NaT,0.0,True
23,CHK-3024,MEM-0019,Frankenstein,Gothic,2026-07-07,2026-07-28,NaT,0.0,True
25,CHK-3026,MEM-0005,Frankenstein,Gothic,2026-04-07,2026-04-28,NaT,0.0,True


In [8]:
checkouts_clean.groupby("genre")["late_fee"].sum()/len(checkouts_clean[checkouts_clean["is_returned"] == True])

genre
Adventure             0.223958
Classic Fiction       0.218750
Dystopian             0.307292
Gothic                0.135417
Historical Fiction    0.260417
Name: late_fee, dtype: float64

In [9]:
len(checkouts_clean[checkouts_clean["is_returned"] == True])

48